# Week 1 — Break a Cipher: the assumption *is* the attack

**Lesson plan:** [`../weeks/week-01.md`](../weeks/week-01.md)

The live-demo notebook. The instructor drives it in Session 1A; you rebuild and
extend it in the 1B studio. Pure Python — no lab target, no LLM, no Docker.

> ### The one idea
> A cipher is not broken by cleverness. It is broken by an **assumption the
> designer didn't know they were making** — here, that the plaintext is English,
> so letter frequencies leak. **Name the assumption and you have named the
> attack.** Every break in this course, classical or modern, is this move.

## 1 · A monoalphabetic substitution cipher

Each letter maps to one other letter, consistently. There are `26! ≈ 4×10²⁶`
possible keys — a keyspace so large that brute force is hopeless. That number is a
trap, and this notebook is about why.

In [ ]:
import random, string
from collections import Counter

ALPHABET = string.ascii_uppercase

def make_key(seed):
    """A random substitution: a permutation of the alphabet."""
    rng = random.Random(seed)
    shuffled = list(ALPHABET)
    rng.shuffle(shuffled)
    return {p: c for p, c in zip(ALPHABET, shuffled)}

def encrypt(text, key):
    out = []
    for ch in text.upper():
        out.append(key.get(ch, ch))     # non-letters pass through
    return "".join(out)

def decrypt(text, key):
    inv = {c: p for p, c in key.items()}
    return "".join(inv.get(ch, ch) for ch in text.upper())

import math
keyspace = math.factorial(26)
print(f"keyspace: 26! = {keyspace:.3e}  (~{keyspace:.0e} keys)")
print("At a trillion keys/second, brute force would take "
      f"{keyspace/1e12/60/60/24/365:.0e} years.")
print("\nSo brute force is hopeless. Watch us break it in well under a second anyway.")

## 2 · Encrypt a real message

The plaintext is ordinary English prose. **That is the whole vulnerability**, though
it doesn't look like one yet.

In [ ]:
PLAINTEXT = """
Security through obscurity is the reliance on secrecy of design as the main method
of providing security for a system. A system relying on obscurity may have real
security vulnerabilities, but its owners or designers believe that if the flaws
are not known then attackers will be unlikely to find them. Kerckhoffs argued the
opposite: a system should be secure even if everything about it except the key is
public knowledge. The lesson for this course is that we can publish exactly how an
attack works, because a system whose security depended on your ignorance was
already broken.
""".strip()

KEY = make_key(seed=1)
CIPHERTEXT = encrypt(PLAINTEXT, KEY)

print("CIPHERTEXT (this is all the attacker gets):\n")
print(CIPHERTEXT[:320], "...")

## 3 · The attacker's assumption, made explicit

The attacker doesn't know the key. But they will *assume* the plaintext is
English — and English has a fixed, well-measured letter-frequency fingerprint.

`E T A O I N S H R` are the most common letters, in roughly that order. If the
ciphertext is a substitution of English, its most common *symbols* should line up
with them.

In [ ]:
# Standard English letter frequencies (%), the attacker's prior knowledge.
ENGLISH_FREQ = {
    "E":12.7,"T":9.1,"A":8.2,"O":7.5,"I":7.0,"N":6.7,"S":6.3,"H":6.1,"R":6.0,
    "D":4.3,"L":4.0,"C":2.8,"U":2.8,"M":2.4,"W":2.4,"F":2.2,"G":2.0,"Y":2.0,
    "P":1.9,"B":1.5,"V":1.0,"K":0.8,"J":0.15,"X":0.15,"Q":0.10,"Z":0.07,
}

def letter_counts(text):
    return Counter(ch for ch in text.upper() if ch in ALPHABET)

cipher_counts = letter_counts(CIPHERTEXT)
total = sum(cipher_counts.values())

print(f"{'cipher sym':>10}{'count':>7}{'freq %':>8}   |   English rank")
print("-" * 48)
english_order = sorted(ENGLISH_FREQ, key=ENGLISH_FREQ.get, reverse=True)
for i, (sym, n) in enumerate(cipher_counts.most_common(9)):
    print(f"{sym:>10}{n:>7}{100*n/total:>7.1f}%   |   {english_order[i]} "
          f"({ENGLISH_FREQ[english_order[i]]}%)")

print("\nThe most common cipher symbol is PROBABLY E, the next PROBABLY T, ...")
print("That is a guess, not a key. Frequency alone gets you close, not correct.")

## 4 · Frequency guess → a first, wrong decryption

Map the *k*-th most common cipher symbol to the *k*-th most common English letter
and decrypt. This will be **mostly wrong** — but readably close, which is the tell
that the assumption is right even when the details aren't.

In [ ]:
def frequency_guess_key(ciphertext):
    """Map cipher symbols to English letters purely by frequency rank."""
    cipher_ranked = [s for s, _ in letter_counts(ciphertext).most_common()]
    eng_ranked = sorted(ENGLISH_FREQ, key=ENGLISH_FREQ.get, reverse=True)
    # decryption map: cipher symbol -> guessed plaintext letter
    guess = {c: e for c, e in zip(cipher_ranked, eng_ranked)}
    return guess

def apply_guess(ciphertext, guess):
    return "".join(guess.get(ch, ch) for ch in ciphertext.upper())

guess = frequency_guess_key(CIPHERTEXT)
attempt = apply_guess(CIPHERTEXT, guess)
print("first frequency-only decryption:\n")
print(attempt[:320], "...")

correct = sum(a == b for a, b in zip(attempt, PLAINTEXT.upper()))
print(f"\ncharacters correct: {correct}/{len(PLAINTEXT)} "
      f"({100*correct/len(PLAINTEXT):.0f}%)")
print("Garbled — but you can SEE English trying to surface. The prior is right;")
print("the per-letter assignment is noisy on a short text. Now we refine.")

## 5 · Refine with a language model — the 1930s kind

Single-letter frequencies are noisy on short text. **Bigrams** (letter pairs) carry
far more structure: `TH`, `HE`, `IN`, `ER` are everywhere in English; `QZ`, `JX`
never appear. We score a candidate decryption by how English-like its bigrams are,
then hill-climb the key: swap two letters, keep the swap if the score improves.

This is a tiny statistical language model driving a local search — the same shape
as an optimizer, decades before anyone called it that.

In [ ]:
# A compact bigram log-likelihood from the plaintext-domain of English. We build
# it from a sample of English so the attack uses only PUBLIC knowledge about the
# language, never the secret plaintext.
ENGLISH_SAMPLE = (PLAINTEXT + " " + """
the quick brown fox jumps over the lazy dog and then the cat sat on the mat while
the rain in spain stays mainly in the plain a system should be secure even if all
about it except the key is public and attackers will study every message they can
""").upper()

def bigram_logscores(sample):
    counts = Counter()
    letters = [c for c in sample if c in ALPHABET]
    for a, b in zip(letters, letters[1:]):
        counts[a + b] += 1
    total = sum(counts.values())
    # log-prob with add-1 smoothing so unseen bigrams are very unlikely, not impossible
    import math
    return counts, total, math.log(1 / (total + 26*26))

BG_COUNTS, BG_TOTAL, BG_FLOOR = bigram_logscores(ENGLISH_SAMPLE)

import math
def score(text):
    """Higher = more English-like, by bigram log-likelihood."""
    letters = [c for c in text.upper() if c in ALPHABET]
    s = 0.0
    for a, b in zip(letters, letters[1:]):
        c = BG_COUNTS.get(a + b, 0)
        s += math.log((c + 1) / (BG_TOTAL + 26*26))
    return s

# Score PER CHARACTER so the comparison is fair regardless of length. (The crack
# below compares decryptions of the SAME ciphertext, so it can use the raw total —
# but for an apples-to-apples demo here we normalize.)
def per_char(text):
    letters = [c for c in text.upper() if c in ALPHABET]
    return score(text) / max(1, len(letters) - 1)

print("English-likeness, per character (higher = more English-like):")
print(f"  real English : {per_char('THE SYSTEM SHOULD BE SECURE EVEN IF IT IS PUBLIC'):.2f}")
print(f"  gibberish    : {per_char('XQZ JKVB WPFBM GYZ ZQXJ KVWP FBMG YZZ QXVB'):.2f}")
print("The gap is small per pair but COMPOUNDS over hundreds of letters — which is")
print("why it can rank 4e26 keys without ever enumerating them.")

In [ ]:
def crack(ciphertext, restarts=8, iters=3000, seed=0):
    """Hill-climb the decryption key to maximize English bigram score."""
    rng = random.Random(seed)
    cipher_syms = [s for s, _ in letter_counts(ciphertext).most_common()]
    best_overall, best_key_overall = None, None

    for r in range(restarts):
        # start from the frequency guess (a good initialization), then perturb
        key = frequency_guess_key(ciphertext)
        # ensure it's a full permutation over the alphabet
        used = set(key.values())
        spare = [e for e in ALPHABET if e not in used]
        for c in ALPHABET:
            if c not in key:
                key[c] = spare.pop()
        current = score(apply_guess(ciphertext, key))
        for _ in range(iters):
            a, b = rng.sample(ALPHABET, 2)          # swap two plaintext letters
            key[a], key[b] = key[b], key[a]
            trial = score(apply_guess(ciphertext, key))
            if trial > current:
                current = trial                     # keep the improving swap
            else:
                key[a], key[b] = key[b], key[a]     # revert
        if best_overall is None or current > best_overall:
            best_overall, best_key_overall = current, dict(key)
    return best_key_overall

cracked_key = crack(CIPHERTEXT, seed=1)
recovered = apply_guess(CIPHERTEXT, cracked_key)

correct = sum(a == b for a, b in zip(recovered, PLAINTEXT.upper()))
print(f"recovered {100*correct/len(PLAINTEXT):.0f}% of characters WITHOUT the key\n")
print(recovered[:320], "...")

## 6 · We never had the key

Confirm the point: the crack used only the ciphertext and public knowledge of
English. It never touched `KEY`. A 4×10²⁶ keyspace fell in a fraction of a second.

In [ ]:
hit = sum(a == b for a, b in zip(recovered, PLAINTEXT.upper()))
print(f"final recovery: {100*hit/len(PLAINTEXT):.0f}% of characters")
print(f"keyspace we did NOT brute-force: 26! = {math.factorial(26):.2e}")
print()
print("The huge keyspace was a decoy. We never searched it. We searched the much")
print("smaller space of 'keys that produce English', guided by letter statistics.")
print()
if hit/len(PLAINTEXT) > 0.9:
    print("Assertion: recovery > 90% — the attack worked.")
    assert hit/len(PLAINTEXT) > 0.9

### ⚠️ Name the assumption, name the attack

The break rested on **one assumption**: the plaintext is English, so its letter
and bigram statistics leak through a substitution.

That tells you exactly how to *defeat* frequency analysis — change what the
assumption relies on:

| Defense | Why it works | What it costs |
|---|---|---|
| Compress before encrypting | flattens letter frequencies | complexity; still not secure alone |
| Encrypt bytes, not letters | no linguistic structure to leak | doesn't help if plaintext is still structured |
| Use a **polyalphabetic** cipher (Vigenère) | one plaintext letter → many cipher letters | broken too, by finding the key length (week 2) |
| One-time pad | ciphertext is statistically independent of plaintext | key as long as the message; **perfect secrecy**, week 2 |

Every one of these is a response to a *named* assumption. This is the habit the
whole course runs on — scorecard **axis 1** (what is the adversary's assumption?)
and **axis 2** (what does the defense guarantee, and under what condition?).

## 7 · The bridge to the rest of the course

What you just built is the template for every attack this term:

1. A **model** of the target's assumptions (English is structured)
2. A **scoring function** for how close a guess is (bigram likelihood)
3. A **search** guided by that score (hill-climbing the key)
4. A **confirmation** that the result is real (readable plaintext, % recovered)

In week 6 the "assumption" is *input is trusted*; in week 9 it's *the model can
tell instructions from data*. Different targets, same move — **find the unstated
assumption and turn it into the attack.**

> Kerckhoffs, 1883: a system should be secure even if everything about it except
> the key is public. We just published the entire attack, and the cipher is no
> weaker for it — because it was already broken by its assumption. That is why this
> course can teach offense openly.

## 8 · Your studio deliverable

In `week01/`:

1. **Break the provided ciphertext** — recover the plaintext, and report the % you
   got before any hand-correction.
2. **Name the assumption** your attack relied on, in one sentence.
3. **Defeat your own attack** — encrypt a message so frequency analysis fails, and
   explain in the Control Scorecard's terms what your defense *guarantees* and what
   it costs (axis 2). "It's harder now" is not a guarantee.
4. **Failure Atlas entry** — the most instructive case is a short ciphertext where
   frequency analysis gets it *wrong*, and why (small samples don't match the
   population frequencies).